# Leisure Centre: Customer Segmentation & Churn Prediction

**Dataset:** Leisure Centre Customer Behaviour Dataset 2024 (synthetic)
**Author:** Dani Baumgarten
**Goal:** Identify distinct customer segments and predict which customers are at risk of not returning.

---

## Project Overview

This notebook follows a two-track pipeline:

1. **Segmentation** - group customers behaviourally using KMeans clustering, so we understand *who* we're dealing with
2. **Churn Prediction** - build a Random Forest classifier to flag customers unlikely to return, so the centre can act on it

The two tracks feed into each other: knowing *which segment churns most* is far more actionable than knowing churn rate in aggregate.

> **Note on the data:** All stats are for the full 2024 calendar year. For churn, we'll simulate a real-world scenario: imagine it's 30 September 2024, and we want to predict who won't come back in Q4.


## 0. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)

import warnings
warnings.filterwarnings('ignore')

SEED = 42
print("✓ All libraries loaded")

In [ ]:
import os

# Diagnostic: show everything Kaggle can see
print("=== Full file tree under /kaggle/input/ ===")
for root, dirs, files in os.walk('/kaggle/input/'):
    level = root.replace('/kaggle/input/', '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root) or 'input'}/")
    for f in files:
        print(f"{indent}  {f}")

## 1. Exploratory Data Analysis

Before modelling, we want to understand the shape of the data: who our customers are, when they visit, how much they spend, and what activities they prefer.

> **Why EDA first?** Jumping straight into a model is tempting but risky. EDA surfaces data quality issues, informs feature choices, and often reveals the most interesting business questions hiding in plain sight.


In [ ]:
# Quick sanity check on the customers table
customers.info()

In [ ]:
customers.describe(include='all').T

### 1a. Who are our customers?

In [ ]:
segment_counts = customers['customer_profile'].value_counts().reset_index()
segment_counts.columns = ['Segment', 'Count']

fig = px.bar(
    segment_counts, x='Segment', y='Count',
    color='Segment',
    title='Number of Customers by Segment',
    template='plotly_white',
    text='Count'
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, xaxis_title='', yaxis_title='Customers')
fig.show()

In [ ]:
fig = px.histogram(
    customers, x='visit_frequency_band',
    color='customer_profile',
    barmode='group',
    title='Visit Frequency Band by Segment',
    template='plotly_white',
    category_orders={
        'visit_frequency_band': ['Rare', 'Occasional', 'Regular', 'Loyal']
    },
    labels={'visit_frequency_band': 'Frequency Band', 'count': 'Customers', 'customer_profile': 'Segment'}
)
fig.show()

### 1b. When do people visit?

Understanding temporal demand patterns matters because a churn model trained without seasonal context could confuse "hasn't visited yet this season" with "won't return ever".


In [ ]:
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']

monthly = (
    visits.groupby('visit_month_name')
    .agg(visit_parties=('visit_id','count'), total_people=('party_size','sum'))
    .reindex(month_order)
    .reset_index()
)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Visit Parties per Month', 'People Through the Door per Month'))

fig.add_trace(
    go.Bar(x=monthly['visit_month_name'], y=monthly['visit_parties'],
           marker_color='steelblue', name='Parties'),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=monthly['visit_month_name'], y=monthly['total_people'],
           marker_color='coral', name='People'),
    row=1, col=2
)
fig.update_layout(title_text='Demand Over Time (2024)', template='plotly_white', showlegend=False)
fig.show()

In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

daily = (
    visits.groupby('day_of_week')
    .agg(avg_visits=('visit_id','count'))
    .reindex(day_order)
    .reset_index()
)

fig = px.bar(
    daily, x='day_of_week', y='avg_visits',
    title='Total Visits by Day of Week',
    template='plotly_white',
    color='avg_visits',
    color_continuous_scale='Blues',
    labels={'day_of_week': '', 'avg_visits': 'Total Visits'}
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

### 1c. Revenue & Spend Patterns

In [ ]:
fig = px.box(
    customers, x='customer_profile', y='total_net_spend_2024_eur',
    color='customer_profile',
    title='Annual Net Spend Distribution by Segment',
    template='plotly_white',
    labels={'customer_profile': 'Segment', 'total_net_spend_2024_eur': 'Total Net Spend 2024 (€)'}
)
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Revenue by activity — which activity drives the most money?
activity_rev = (
    transactions.groupby('activity_type')['net_amount_eur']
    .sum()
    .reset_index()
    .sort_values('net_amount_eur', ascending=False)
)
activity_rev.columns = ['Activity', 'Net Revenue (€)']

fig = px.bar(
    activity_rev, x='Activity', y='Net Revenue (€)',
    color='Activity',
    title='Total Net Revenue by Activity (2024)',
    template='plotly_white',
    text=activity_rev['Net Revenue (€)'].apply(lambda x: f'€{x:,.0f}')
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False)
fig.show()

## 2. Defining Churn

> **This is the most important modelling decision you'll make.** There's no universal definition of churn — it depends on the business context. Get this wrong, and the model is answering the wrong question.

**Our definition:**
A customer is considered *churned* if their `last_visit_date` is before **1 October 2024**, meaning they didn't return in Q4.

**Why this cutoff?**
- Q4 includes peak season (December, school holidays) — if a customer skips it, that's a meaningful signal
- It gives us 9 months of history to train on, 3 months to predict
- It mirrors a real-world scenario: "It's the end of September — who should we send a win-back campaign to?"

**A note of caution:** Because the dataset covers the full year, some of our features (like `total_net_spend_2024_eur`) include Q4 data, which would be unavailable in a real system at the cutoff point. In a production pipeline, you'd filter visits to pre-cutoff only. For this portfolio project, we'll flag this and move on — it's a common and well-understood trade-off.


In [ ]:
CUTOFF = pd.Timestamp('2024-09-30')

customers['churned'] = (customers['last_visit_date'] < CUTOFF).astype(int)

churn_rate = customers['churned'].mean()
n_churned  = customers['churned'].sum()
n_retained = len(customers) - n_churned

print(f"Cutoff date     : {CUTOFF.date()}")
print(f"Total customers : {len(customers):,}")
print(f"Churned (no Q4 visit) : {n_churned:,}  ({churn_rate:.1%})")
print(f"Retained (returned)   : {n_retained:,}  ({1 - churn_rate:.1%})")

In [ ]:
# Class balance check — important before any classification task
fig = px.pie(
    values=[n_retained, n_churned],
    names=['Retained', 'Churned'],
    title='Churn vs Retained Customers',
    color_discrete_sequence=['#2ecc71', '#e74c3c'],
    template='plotly_white'
)
fig.update_traces(textinfo='percent+label')
fig.show()

### Class Imbalance?

If churn rate is very low (e.g. <10%), we'd need to handle imbalance with techniques like SMOTE or class weighting. Random Forest has a `class_weight='balanced'` parameter that adjusts automatically — we'll use that.

Check the chart above and note the ratio. If it's roughly 40–60%, we're in good shape.


## 3. Feature Engineering

Feature engineering is where domain knowledge becomes model fuel. We're selecting and transforming raw columns into a format the model can learn from.

**Features we'll use:**

| Feature | Type | Why it matters |
|---|---|---|
| `total_visits_2024` | Numeric | Frequency — core loyalty signal |
| `total_net_spend_2024_eur` | Numeric | Monetary value |
| `average_visit_spend_eur` | Numeric | Spend per trip |
| `average_party_size` | Numeric | Group behaviour |
| `online_visit_share` | Numeric | Channel preference |
| `form_completion_rate` | Numeric | Engagement proxy |
| `newsletter_subscribed` | Binary | Marketing engagement |
| `customer_profile` | Categorical | Segment identity |
| `preferred_activity` | Categorical | Behavioural preference |
| `preferred_visit_time` | Categorical | Time-of-day preference |
| `home_area` | Categorical | Geography / accessibility |

**Categorical encoding:** We'll use `pd.get_dummies()` (one-hot encoding). This creates a binary column for each category value, which tree-based models like Random Forest handle well.


In [ ]:
NUMERIC_FEATURES = [
    'total_visits_2024',
    'total_net_spend_2024_eur',
    'average_visit_spend_eur',
    'average_party_size',
    'online_visit_share',
    'form_completion_rate',
    'newsletter_subscribed',
]

CAT_FEATURES = [
    'customer_profile',
    'preferred_activity',
    'preferred_visit_time',
    'home_area',
]

df_model = customers[NUMERIC_FEATURES + CAT_FEATURES + ['churned']].copy()

# One-hot encode categoricals
df_encoded = pd.get_dummies(df_model, columns=CAT_FEATURES, drop_first=False)

X = df_encoded.drop('churned', axis=1)
y = df_encoded['churned']

print(f"Feature matrix shape : {X.shape}")
print(f"Features             : {list(X.columns)}")

In [ ]:
# Check for any missing values
missing = X.isnull().sum()
print("Missing values per feature:")
print(missing[missing > 0] if missing.any() else "None — all clean ✓")

## 4. Customer Segmentation with KMeans

### What is KMeans?

KMeans is an *unsupervised* algorithm — it finds natural groupings in the data without being told what the groups are. It works by:

1. Randomly placing `k` cluster centres in feature space
2. Assigning each customer to their nearest centre
3. Moving each centre to the mean of its assigned customers
4. Repeating steps 2–3 until the centres stop moving

**Key decision: how many clusters (k)?**
We'll use the *elbow method* — plot inertia (sum of squared distances from each point to its cluster centre) across a range of k values. The "elbow" in the curve is where adding more clusters stops meaningfully reducing inertia.

> **Why segment before predicting churn?** Segmentation gives us *interpretable levers*. Saying "Cluster 2 has an 80% churn rate and consists of high-spend occasional visitors" is far more actionable than a raw churn probability.


In [ ]:
# For clustering we use the numeric features only — easier to interpret
CLUSTER_FEATURES = [
    'total_visits_2024',
    'total_net_spend_2024_eur',
    'average_party_size',
    'online_visit_share',
    'form_completion_rate',
    'newsletter_subscribed',
]

X_cluster = customers[CLUSTER_FEATURES].copy()

# StandardScaler normalises each feature to mean=0, std=1
# This is ESSENTIAL for KMeans — otherwise features with larger scales dominate
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

print("Features scaled. Example — first row:")
print(dict(zip(CLUSTER_FEATURES, X_scaled[0].round(2))))

In [ ]:
# Elbow Method
inertia = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

fig = px.line(
    x=list(K_range), y=inertia,
    markers=True,
    title='Elbow Method — Finding the Optimal Number of Clusters',
    labels={'x': 'Number of Clusters (k)', 'y': 'Inertia (Within-cluster Sum of Squares)'},
    template='plotly_white'
)
fig.update_traces(line_color='steelblue', marker_size=8)
fig.add_annotation(text="← Look for the 'elbow' here", x=4, y=inertia[2],
                   showarrow=True, arrowhead=2, ax=60, ay=-40)
fig.show()

**How to read the elbow plot:** Look for the point where the curve bends sharply — that's where additional clusters stop buying much. Typically k=4 or k=5 is a sweet spot for a dataset this size.


In [ ]:
# Fit final KMeans — adjust K if the elbow suggests differently
K = 4

km_final = KMeans(n_clusters=K, random_state=SEED, n_init=10)
customers['cluster'] = km_final.fit_predict(X_scaled)

print(f"Cluster assignments (n per cluster):")
print(customers['cluster'].value_counts().sort_index().to_string())

### Visualising Clusters with PCA

With 6 clustering features, we can't plot clusters directly. PCA (Principal Component Analysis) compresses them into 2 dimensions while preserving as much variance as possible — perfect for a scatter plot.

> PCA is a *dimensionality reduction* technique, not a model. Think of it as finding the "most informative angle" to view the data from.


In [ ]:
pca = PCA(n_components=2, random_state=SEED)
components = pca.fit_transform(X_scaled)

customers['pc1'] = components[:, 0]
customers['pc2'] = components[:, 1]

var1 = pca.explained_variance_ratio_[0]
var2 = pca.explained_variance_ratio_[1]

fig = px.scatter(
    customers,
    x='pc1', y='pc2',
    color=customers['cluster'].astype(str),
    hover_data={
        'customer_profile': True,
        'total_visits_2024': True,
        'total_net_spend_2024_eur': True,
        'pc1': False, 'pc2': False
    },
    title=f'Customer Clusters (PCA Projection) — {var1+var2:.1%} of variance explained',
    labels={
        'pc1': f'PC1 ({var1:.1%} variance)',
        'pc2': f'PC2 ({var2:.1%} variance)',
        'color': 'Cluster'
    },
    template='plotly_white',
    opacity=0.65,
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_traces(marker_size=5)
fig.show()

### Cluster Profiles

Now the fun part — giving each cluster a name based on its characteristics.


In [ ]:
profile = (
    customers.groupby('cluster')
    .agg(
        n_customers=('churned', 'count'),
        avg_visits=('total_visits_2024', 'mean'),
        avg_spend=('total_net_spend_2024_eur', 'mean'),
        avg_party_size=('average_party_size', 'mean'),
        online_share=('online_visit_share', 'mean'),
        newsletter_rate=('newsletter_subscribed', 'mean'),
        churn_rate=('churned', 'mean')
    )
    .round(2)
    .reset_index()
)

profile.columns = [
    'Cluster', 'Customers', 'Avg Visits', 'Avg Spend (€)',
    'Avg Party Size', 'Online Share', 'Newsletter Rate', 'Churn Rate'
]

print(profile.to_string(index=False))

In [ ]:
# Visualise churn rate per cluster
fig = px.bar(
    profile, x='Cluster', y='Churn Rate',
    color='Churn Rate',
    text=profile['Churn Rate'].apply(lambda x: f'{x:.1%}'),
    title='Churn Rate by Cluster',
    template='plotly_white',
    color_continuous_scale='RdYlGn_r',
    labels={'Churn Rate': 'Churn Rate (Q4 non-return)'}
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False, yaxis_tickformat='.0%')
fig.show()

In [ ]:
# Most common segment per cluster — helps us label the clusters
seg_by_cluster = (
    customers.groupby(['cluster', 'customer_profile'])
    .size()
    .reset_index(name='count')
    .sort_values(['cluster', 'count'], ascending=[True, False])
)
top_seg = seg_by_cluster.groupby('cluster').first().reset_index()
print("Dominant segment per cluster:")
print(top_seg[['cluster', 'customer_profile', 'count']].to_string(index=False))

## 5. Churn Prediction with Random Forest

### What is a Random Forest?

A Random Forest is an *ensemble* of decision trees. Each tree is trained on a random sample of the data and a random subset of features. The final prediction is a majority vote across all trees.

**Why Random Forest for this problem?**
- Handles mixed feature types (numeric + one-hot encoded) well
- Robust to outliers and doesn't require feature scaling
- Built-in feature importance scores
- `class_weight='balanced'` handles class imbalance automatically
- Generally performs well out of the box with minimal tuning

> **Train/test split:** We'll hold out 20% of customers for evaluation. The model never sees these during training — they're our honest estimate of real-world performance.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y  # ensures the churn ratio is the same in train and test
)

print(f"Training set : {X_train.shape[0]:,} customers  |  churn rate: {y_train.mean():.1%}")
print(f"Test set     : {X_test.shape[0]:,} customers  |  churn rate: {y_test.mean():.1%}")

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,    # 200 trees — more = better, with diminishing returns
    max_depth=8,         # prevents individual trees from memorising the data
    class_weight='balanced',  # compensates if one class is more common
    random_state=SEED,
    n_jobs=-1            # use all CPU cores
)

rf.fit(X_train, y_train)
print("✓ Model trained")

In [ ]:
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]  # probability of churning

print(classification_report(y_test, y_pred, target_names=['Retained (0)', 'Churned (1)']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

### Reading the Classification Report

- **Precision** — of everyone the model flagged as churned, what % actually churned? (avoids false alarms)
- **Recall** — of everyone who actually churned, what % did the model catch? (avoids missing churners)
- **F1-score** — harmonic mean of precision and recall; useful when classes are imbalanced
- **ROC-AUC** — overall discrimination ability, 0.5 = random, 1.0 = perfect

For a churn model, **recall is usually more important** — missing a churner who could have been retained is costly.


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig = px.imshow(
    cm,
    labels=dict(x='Predicted', y='Actual', color='Count'),
    x=['Retained', 'Churned'],
    y=['Retained', 'Churned'],
    text_auto=True,
    color_continuous_scale='Blues',
    title='Confusion Matrix',
    template='plotly_white'
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fpr, y=tpr, mode='lines',
    name=f'Random Forest (AUC = {auc:.3f})',
    line=dict(color='royalblue', width=2.5)
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines',
    name='Random Classifier (AUC = 0.500)',
    line=dict(color='lightgrey', dash='dash')
))
fig.update_layout(
    title='ROC Curve — Churn Prediction Model',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate (Recall)',
    template='plotly_white',
    legend=dict(x=0.55, y=0.05)
)
fig.show()

### Feature Importance

Random Forest tells us which features it relied on most. This isn't just a model diagnostic — it tells us what actually *drives* churn in this dataset.


In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns)
top15 = importances.nlargest(15).sort_values()

fig = px.bar(
    x=top15.values,
    y=top15.index,
    orientation='h',
    title='Top 15 Features — Random Forest Feature Importance',
    labels={'x': 'Importance Score', 'y': ''},
    template='plotly_white',
    color=top15.values,
    color_continuous_scale='Blues'
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 6. Segment × Churn Analysis

Now we connect the two threads: which customer segments and clusters are churning most, and what does that tell us operationally?


In [ ]:
# Churn rate by customer profile
seg_churn = (
    customers.groupby('customer_profile')['churned']
    .agg(['mean', 'sum', 'count'])
    .reset_index()
)
seg_churn.columns = ['Segment', 'Churn Rate', 'Churned', 'Total']
seg_churn = seg_churn.sort_values('Churn Rate', ascending=True)

fig = px.bar(
    seg_churn, x='Churn Rate', y='Segment',
    orientation='h',
    color='Churn Rate',
    text=seg_churn['Churn Rate'].apply(lambda x: f'{x:.1%}'),
    title='Churn Rate by Customer Segment',
    template='plotly_white',
    color_continuous_scale='RdYlGn_r',
    labels={'Churn Rate': 'Did not return in Q4'}
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False, xaxis_tickformat='.0%', yaxis_title='')
fig.show()

In [ ]:
# Churn rate by preferred activity
act_churn = (
    customers.groupby('preferred_activity')['churned']
    .agg(['mean', 'count'])
    .reset_index()
)
act_churn.columns = ['Activity', 'Churn Rate', 'Customers']
act_churn = act_churn.sort_values('Churn Rate', ascending=True)

fig = px.bar(
    act_churn, x='Activity', y='Churn Rate',
    color='Churn Rate',
    text=act_churn['Churn Rate'].apply(lambda x: f'{x:.1%}'),
    title='Churn Rate by Preferred Activity',
    template='plotly_white',
    color_continuous_scale='RdYlGn_r',
    labels={'Churn Rate': 'Churn Rate'}
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False, yaxis_tickformat='.0%')
fig.show()

In [ ]:
# Newsletter impact: do subscribers churn less?
newsletter_churn = (
    customers.groupby('newsletter_subscribed')['churned']
    .agg(['mean', 'count'])
    .reset_index()
)
newsletter_churn['newsletter_subscribed'] = newsletter_churn['newsletter_subscribed'].map({0: 'Not Subscribed', 1: 'Subscribed'})
newsletter_churn.columns = ['Newsletter', 'Churn Rate', 'Customers']

fig = px.bar(
    newsletter_churn, x='Newsletter', y='Churn Rate',
    color='Newsletter',
    text=newsletter_churn['Churn Rate'].apply(lambda x: f'{x:.1%}'),
    title='Churn Rate: Newsletter Subscribers vs Non-Subscribers',
    template='plotly_white',
    color_discrete_sequence=['#e74c3c', '#2ecc71'],
    labels={'Churn Rate': 'Churn Rate', 'Newsletter': ''}
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, yaxis_tickformat='.0%')
fig.show()

In [ ]:
# Churn by frequency band
freq_churn = (
    customers.groupby('visit_frequency_band')['churned']
    .agg(['mean', 'count'])
    .reset_index()
)
freq_churn.columns = ['Frequency Band', 'Churn Rate', 'Customers']
freq_order = ['Rare', 'Occasional', 'Regular', 'Loyal']
freq_churn['Frequency Band'] = pd.Categorical(freq_churn['Frequency Band'], categories=freq_order, ordered=True)
freq_churn = freq_churn.sort_values('Frequency Band')

fig = px.bar(
    freq_churn, x='Frequency Band', y='Churn Rate',
    color='Churn Rate',
    text=freq_churn['Churn Rate'].apply(lambda x: f'{x:.1%}'),
    title='Churn Rate by Visit Frequency Band',
    template='plotly_white',
    color_continuous_scale='RdYlGn_r',
    labels={'Churn Rate': 'Churn Rate', 'Frequency Band': ''}
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False, yaxis_tickformat='.0%')
fig.show()

In [ ]:
# Add churn probability back to customer table for exploration
test_indices = X_test.index
customers.loc[test_indices, 'churn_probability'] = y_prob

# Top 20 highest-risk customers in test set
high_risk = (
    customers.loc[test_indices]
    .sort_values('churn_probability', ascending=False)
    [['customer_profile', 'visit_frequency_band', 'preferred_activity',
      'total_visits_2024', 'total_net_spend_2024_eur', 'newsletter_subscribed',
      'cluster', 'churn_probability']]
    .head(20)
)
print("Top 20 highest churn-risk customers (test set):")
high_risk

## 7. Key Takeaways & Next Steps

### What the model tells us

1. **Feature importance** reveals the true drivers of churn — compare the top features to what you'd have guessed intuitively. Surprising?

2. **Newsletter subscribers churn at a meaningfully lower rate** — a simple correlation, but it reinforces the value of the email list as a retention tool.

3. **Cluster profiling** shows which behavioural groups are highest risk — this is where campaign targeting gets interesting.

4. **Frequency band is highly predictive** — Rare visitors churn overwhelmingly. This suggests the first 1–2 visits are the make-or-break window.

---

### Where to go next

Here are some natural extensions to push this project further:

- **Hyperparameter tuning** — try `GridSearchCV` or `RandomizedSearchCV` on the Random Forest, or swap in XGBoost and compare AUC scores
- **RFM Scoring** — formally compute Recency / Frequency / Monetary scores and use them as features or as a segmentation layer
- **SHAP values** — move beyond global feature importance to *per-customer* explanations of why the model flagged each individual as high risk
- **Survival analysis** — instead of binary churn, model *time until churn* using Kaplan-Meier curves
- **A/B test simulation** — use the model's risk scores to simulate a win-back campaign and estimate its revenue impact

---

> **Challenge for you:** Look at the high-risk customer table. Can you write a short brief — 3 bullet points — that a marketing manager could act on right now? That's the bridge from data science to business value. 🎯
